In [ ]:
import os
os.chdir('/kaggle/working')

!git clone https://github.com/basiaseweryn/generative-models-cat-images.git

In [ ]:
!git pull

In [ ]:
%cd /kaggle/working/generative-models-cat-images

In [ ]:
import sys
import os

PROJECT_ROOT = "/kaggle/working/generative-models-cat-images"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

UTILS_PATH = os.path.join(PROJECT_ROOT, "utils")
if UTILS_PATH not in sys.path:
    sys.path.insert(0, UTILS_PATH)

print(f"project root added: {PROJECT_ROOT}")
print(f"utils folder added: {UTILS_PATH}")
print(f"available files: {os.listdir(PROJECT_ROOT)}")

In [ ]:
import os
import torch
import torch.optim as optim
from tqdm.auto import tqdm

from utils.config import (
    EXPERIMENTS,
    FID_NUM_SAMPLES,
    RANDOM_SEED,
    STAGE_1_SCENARIOS,
    STAGE_2_SCENARIOS,
    STAGE_4_SCENARIOS,
    TRAINED_MODELS_DIR,
    seed_everything,
)
from utils.data_utils import get_dataloader
from utils.models import get_model, sample_dcgan, sample_vae, vae_loss
from utils.evaluation_utils import (
    compare_cats_vs_cats_dogs,
    compare_fid_scenarios,
    compare_mode_collapse_scenarios,
    denormalize_dcgan,
    evaluate_fid_scenario,
    generate_candidate_grid,
    interpolate_between_latents,
    load_trained_model,
    output_path,
    print_fid_table,
    save_checkpoint,
    save_interpolation_grid,
    save_latent_codes,
    save_sample_grid,
    show_image_grid,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(RANDOM_SEED)
print("Device:", device)

## Training loops

In [ ]:
def train_vae_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, total_recon, total_kl = 0.0, 0.0, 0.0
    for batch in tqdm(loader, leave=False):
        if isinstance(batch, (list, tuple)):
            batch = batch[0]
        batch = batch.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = model(batch)
        loss, recon_l, kl_l = vae_loss(recon, batch, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_recon += recon_l.item()
        total_kl += kl_l.item()
    n = len(loader)
    return total_loss / n, total_recon / n, total_kl / n


def train_dcgan_one_epoch(model, loader, opt_g, opt_d, device, config):
    model.train()
    n_critic = config.get("n_critic", 1)
    noise_std = config.get("instance_noise_std", 0.0)
    latent_dim = config.get("latent_dim", 128)
    criterion = torch.nn.BCELoss()
    g_losses, d_losses = [], []

    for batch in tqdm(loader, leave=False):
        if isinstance(batch, (list, tuple)):
            batch = batch[0]
        real = batch.to(device)
        bsz = real.size(0)
        real_labels = torch.ones(bsz, 1, device=device)
        fake_labels = torch.zeros(bsz, 1, device=device)

        for _ in range(n_critic):
            opt_d.zero_grad()
            z = torch.randn(bsz, latent_dim, device=device)
            fake = model.generator(z).detach()
            real_in = real + torch.randn_like(real) * noise_std if noise_std > 0 else real
            fake_in = fake + torch.randn_like(fake) * noise_std if noise_std > 0 else fake
            loss_d = (
                criterion(model.discriminator(real_in), real_labels)
                + criterion(model.discriminator(fake_in), fake_labels)
            ) * 0.5
            loss_d.backward()
            opt_d.step()
            d_losses.append(loss_d.item())

        opt_g.zero_grad()
        z = torch.randn(bsz, latent_dim, device=device)
        fake = model.generator(z)
        loss_g = criterion(model.discriminator(fake), real_labels)
        loss_g.backward()
        opt_g.step()
        g_losses.append(loss_g.item())

    return sum(g_losses) / len(g_losses), sum(d_losses) / len(d_losses)


def run_training(scenario_name: str):
    config = EXPERIMENTS[scenario_name].copy()
    loader = get_dataloader(config)
    model = get_model(config["model"], config).to(device)
    save_dir = os.path.join(TRAINED_MODELS_DIR, scenario_name)
    os.makedirs(save_dir, exist_ok=True)
    history = {"epoch": [], "loss": [], "recon": [], "kl": [], "g_loss": [], "d_loss": []}

    if config["model"] == "vae":
        optimizer = optim.Adam(model.parameters(), lr=config["lr"])
        for epoch in range(1, config["epochs"] + 1):
            loss, recon, kl = train_vae_one_epoch(model, loader, optimizer, device)
            history["epoch"].append(epoch)
            history["loss"].append(loss)
            history["recon"].append(recon)
            history["kl"].append(kl)
            print(f"Epoch {epoch}/{config['epochs']} | loss={loss:.4f} recon={recon:.4f} kl={kl:.4f}")
            if epoch % 10 == 0 or epoch == config["epochs"]:
                samples = sample_vae(model, 64, device)
                save_sample_grid(samples, os.path.join(save_dir, f"samples_epoch_{epoch}.png"))
        save_checkpoint(os.path.join(save_dir, "model.pth"), model, optimizer, config["epochs"], history, config)
        return model, history, config

    opt_g = optim.Adam(model.generator.parameters(), lr=config["lr"], betas=(0.5, 0.999))
    opt_d = optim.Adam(model.discriminator.parameters(), lr=config["lr"], betas=(0.5, 0.999))
    for epoch in range(1, config["epochs"] + 1):
        g_loss, d_loss = train_dcgan_one_epoch(model, loader, opt_g, opt_d, device, config)
        history["epoch"].append(epoch)
        history["g_loss"].append(g_loss)
        history["d_loss"].append(d_loss)
        print(f"Epoch {epoch}/{config['epochs']} | G={g_loss:.4f} D={d_loss:.4f}")
        if epoch % 10 == 0 or epoch == config["epochs"]:
            samples = sample_dcgan(model.generator, 64, config["latent_dim"], device)
            save_sample_grid(denormalize_dcgan(samples), os.path.join(save_dir, f"samples_epoch_{epoch}.png"))
    torch.save(
        {"epoch": config["epochs"], "model_state_dict": model.state_dict(),
         "opt_g": opt_g.state_dict(), "opt_d": opt_d.state_dict(), "history": history, "config": config},
        os.path.join(save_dir, "model.pth"),
    )
    return model, history, config

## Stage 1 — Baseline training

Train both models (run this cell twice with different `SCENARIO`, or run both in sequence).

In [ ]:
SCENARIO = "stage_1_vae"  # then "stage_1_dcgan"

model, history, config = run_training(SCENARIO)

## Quantitative comparison — FID (VAE vs DCGAN)

Uses **2000** real/fake samples by default (`FID_NUM_SAMPLES` in config). Lower FID = better.

In [ ]:
# After stage_1_vae and stage_1_dcgan are trained:
fid_rows_stage1 = compare_fid_scenarios(
    STAGE_1_SCENARIOS,
    device,
    max_samples=FID_NUM_SAMPLES,
    save_csv=output_path("reports", "fid_stage1.csv"),
)
print_fid_table(fid_rows_stage1)

## Qualitative assessment — sample grid

In [ ]:
SCENARIO = "stage_1_dcgan"  # model to inspect

model, config, _ = load_trained_model(SCENARIO, device)
display_imgs, latents, _ = generate_candidate_grid(model, config, device, num_candidates=64, seed=42)
show_image_grid(display_imgs, title=f"Generated samples — {SCENARIO}", nrow=8)
save_sample_grid(display_imgs, output_path(SCENARIO, "qualitative_grid.png"), nrow=8)

## Stage 2 — Hyperparameter experiments

Train each `stage_2_*` scenario (subset of data for speed), then run the FID table below.

In [ ]:
# Train one hyperparameter run at a time, e.g.:
# for name in STAGE_2_SCENARIOS:
#     run_training(name)

In [ ]:
fid_rows_stage2 = compare_fid_scenarios(
    STAGE_2_SCENARIOS,
    device,
    max_samples=FID_NUM_SAMPLES,
    save_csv=output_path("reports", "fid_stage2_hyperparams.csv"),
)
print_fid_table(fid_rows_stage2)

# For your report: which lr / latent_dim / batch_size gave the best FID per model?

## Stage 3 — Latent interpolation (10 images)

1. Generate a candidate grid  
2. Pick two indices (`IDX_A`, `IDX_B`)  
3. Save their latent vectors  
4. Linearly interpolate → **8 intermediate** + **2 endpoints** = 10 images  

**Report discussion:** Is the transition smooth? What does that imply about the latent space?

In [ ]:
SCENARIO = "stage_1_vae"  # or your best stage_2 / stage_4 scenario

model, config, _ = load_trained_model(SCENARIO, device)
display_imgs, latents, _ = generate_candidate_grid(model, config, device, num_candidates=64, seed=123)
show_image_grid(display_imgs, title="Pick two images — note their indices", nrow=8)

# Change these after inspecting the grid above
IDX_A = 3
IDX_B = 47

z_a = latents[IDX_A].clone()
z_b = latents[IDX_B].clone()
save_latent_codes(z_a, z_b, output_path(SCENARIO, "interpolation_latents.pth"))

print("z_a shape:", z_a.shape)
print("z_b shape:", z_b.shape)
print("z_a (first 8 values):", z_a[:8].cpu().numpy())

In [ ]:
interp_display, interp_raw, is_dcgan = interpolate_between_latents(
    model, config, z_a, z_b, device, num_steps=8
)

path = output_path(SCENARIO, "interpolation_10_images.png")
save_interpolation_grid(interp_raw, path, is_dcgan=is_dcgan)
show_image_grid(interp_display, title=f"Interpolation: idx {IDX_A} -> {IDX_B} (10 images)", nrow=10)

# Endpoint-only view (the two originally selected generations)
endpoints = torch.stack([interp_display[0], interp_display[-1]])
show_image_grid(endpoints, title="Endpoints (selected generated images)", nrow=2)

## Stage 4 — Mode collapse

If `likely_mode_collapse` is True, retrain with a `stage_4_*` scenario and compare again.

In [ ]:
# Check baseline DCGAN (train stage_4_* if collapse is likely)
collapse_results = compare_mode_collapse_scenarios(STAGE_4_SCENARIOS, device)

# Optional: train mitigations first
# run_training("stage_4_dcgan_n_critic_5")
# run_training("stage_4_dcgan_instance_noise")

In [ ]:
# FID after mitigation — compare to stage_1_dcgan
fid_rows_stage4 = compare_fid_scenarios(
    STAGE_4_SCENARIOS,
    device,
    max_samples=FID_NUM_SAMPLES,
    save_csv=output_path("reports", "fid_stage4_mode_collapse.csv"),
)
print_fid_table(fid_rows_stage4)

## Stage 5 — Cats + dogs vs cats only

Exploratory: do samples look like distinct cats/dogs or blends? Compare FID and grids.

In [ ]:
# Train after picking your best architecture from stage 1–2:
# run_training("stage_5_dcgan_cats_dogs")  # or stage_5_vae_cats_dogs
# run_training("stage_1_dcgan")  # cats-only baseline if not already trained

In [ ]:
SCENARIO_CATS_ONLY = "stage_1_dcgan"
SCENARIO_MIXED = "stage_5_dcgan_cats_dogs"

summary, imgs_cat, imgs_mix = compare_cats_vs_cats_dogs(
    SCENARIO_CATS_ONLY,
    SCENARIO_MIXED,
    device,
    num_samples=64,
)

show_image_grid(imgs_cat, title=f"Cats only — {SCENARIO_CATS_ONLY}", nrow=8)
show_image_grid(imgs_mix, title=f"Cats + dogs — {SCENARIO_MIXED}", nrow=8)

# Report: discuss whether outputs are class-distinct or blended (exploratory).

## Additional findings (for your report)

Use this checklist after experiments:
- VAE vs DCGAN: FID table + sharpness/blur in sample grids
- Best hyperparameters from stage 2 CSV
- Interpolation smoothness (stage 3)
- Mode collapse: diversity scores + whether stage 4 helped
- Cats+dogs: distinct classes vs chimera images
- Compute strategy: 64×64, `subset_ratio`, epoch counts